In [1]:
!git clone https://github.com/vfxbymani/ai-localization-pipeline.git
%cd ai-localization-pipeline
!pip install -q -U google-genai
!ls

Cloning into 'ai-localization-pipeline'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 56 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 1.32 MiB | 12.87 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/kaggle/working/ai-localization-pipeline
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-aiplatform 2.7.0 requires google-genai<3.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 3.0.0 which is incompatible.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.

In [2]:
from kaggle_secrets import UserSecretsClient
from google import genai

api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

names = [m.name for m in client.models.list()]
flash = [n for n in names if "flash" in n.lower()]

print("models visible to this key:", len(names))
for n in flash[:15]:
    print(n)

models visible to this key: 26
models/gemini-2.5-flash
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash


In [3]:
import re
import time
from google.genai import types

def parse_srt(path):
    text = open(path, encoding="utf-8").read().strip()
    cues = []
    for block in re.split(r"\n\s*\n", text):
        lines = block.strip().splitlines()
        if len(lines) < 3:
            continue
        start, end = [t.strip() for t in lines[1].split("-->")]
        cues.append({"index": int(lines[0]), "start": start, "end": end, "text": " ".join(lines[2:])})
    return cues

en_cues = parse_srt("test_video.srt")
ta_cues = parse_srt("test_video_tamil.srt")
assert len(en_cues) == len(ta_cues)

ABBREV = {"mr.", "mrs.", "ms.", "dr.", "st."}
def ends_sentence(text):
    last = text.split()[-1]
    return last[-1] in ".?!" and last.lower() not in ABBREV

groups, cur = [], []
for i, c in enumerate(en_cues):
    cur.append(i)
    if ends_sentence(c["text"]):
        groups.append(cur)
        cur = []
if cur:
    groups.append(cur)

pairs = [
    (" ".join(en_cues[i]["text"] for i in g), " ".join(ta_cues[i]["text"] for i in g))
    for g in groups
]
print("sentences:", len(pairs))

GLOSSARY = {
    "Quilter": "குயில்டர்",
    "Leighton": "லெய்டன்",
    "Layton": "லெய்டன்",
    "Ruskin": "ரஸ்கின்",
    "Michelangelo": "மைக்கலேஞ்சலோ",
    "Mr.": "திரு.",
    "similes": "உவமைகள்",
}

def build_prompt(en, ta):
    gl = "\n".join(f"- {k} -> {v}" for k, v in GLOSSARY.items())
    return f"""You are a professional English-to-Tamil subtitle editor.
Below are an English sentence and a machine translation of it into Tamil.
Fix the Tamil so it is accurate, natural and complete: every idea in the English must be present.

Rules:
- Whenever the English contains a glossary term, write exactly the Tamil form given in the glossary.
- Keep the meaning. Do not add information.
- Write in Tamil script. Do not leave English words in the sentence.
- Output only the corrected Tamil sentence, with no explanation.

Glossary:
{gl}

English: {en}
Machine translation: {ta}
Corrected Tamil:"""

cfg = types.GenerateContentConfig(temperature=0.2)
candidates = ["gemini-2.5-flash", "gemini-3.5-flash-lite", "gemini-3.5-flash"]

def compliance(en, out):
    need = [v for k, v in GLOSSARY.items() if k.lower() in en.lower()]
    return sum(v in out for v in need), len(need)

edits = {m: [] for m in candidates}
for m in candidates:
    for en, ta in pairs[:3]:
        try:
            r = client.models.generate_content(model=m, contents=build_prompt(en, ta), config=cfg)
            edits[m].append(r.text.strip())
        except Exception as e:
            edits[m].append(None)
            print(f"[{m}] ERROR: {type(e).__name__}: {str(e)[:150]}")
        time.sleep(4)

for i, (en, ta) in enumerate(pairs[:3]):
    print("EN:  ", en)
    print("NLLB:", ta)
    for m in candidates:
        out = edits[m][i]
        if out is None:
            print(f"[{m}] no result")
        else:
            ok, total = compliance(en, out)
            print(f"[{m}] glossary {ok}/{total} | {out}")
    print()

sentences: 14
[gemini-2.5-flash] ERROR: ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to 
[gemini-2.5-flash] ERROR: ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to 
[gemini-2.5-flash] ERROR: ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to 
[gemini-3.5-flash] ERROR: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please 
EN:   Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.
NLLB: திரு. குயில்டர் நடுத்தர வர்க்கத்தின் அப்போஸ்தலர், அவருடைய நற்செய்தியை வரவேற்பதில் நாங்கள் மகிழ்ச்சியடைகிறோம்.
[gemini-2.5-fla

In [4]:
!pip install -q rapidfuzz
import json
import re
import time
from rapidfuzz import fuzz

models = ["gemini-3.5-flash-lite", "gemini-3.5-flash"]

def call(model, prompt, tries=3):
    for k in range(tries):
        try:
            r = client.models.generate_content(model=model, contents=prompt, config=cfg)
            return r.text.strip()
        except Exception as e:
            msg = str(e)
            retryable = any(x in msg for x in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if retryable and k < tries - 1:
                time.sleep(15 * (k + 1))
                continue
            print(f"[{model}] ERROR: {type(e).__name__}: {msg[:120]}")
            return None

def stem(v):
    return v.rstrip("\u0bcd")

def glossary_check(en, out):
    need = [v for k, v in GLOSSARY.items() if k.lower() in en.lower()]
    return sum(stem(v) in out for v in need), len(need)

results = {m: [] for m in models}
for m in models:
    for en, ta in pairs:
        results[m].append(call(m, build_prompt(en, ta)))
        time.sleep(4)

with open("postedit_results.json", "w", encoding="utf-8") as f:
    json.dump({"pairs": pairs, "results": results}, f, ensure_ascii=False, indent=1)

def summarize(m):
    gl_ok = gl_all = 0
    sims, ratios, flags = [], [], []
    for i, ((en, ta), o) in enumerate(zip(pairs, results[m])):
        if not o:
            flags.append((i, "no result"))
            continue
        ok, tot = glossary_check(en, o)
        gl_ok, gl_all = gl_ok + ok, gl_all + tot
        sim = fuzz.ratio(ta, o)
        ratio = len(o) / len(ta)
        sims.append(sim)
        ratios.append(ratio)
        why = []
        if ok < tot: why.append("glossary")
        if re.search(r"[A-Za-z]", o): why.append("latin letters")
        if sim < 60: why.append("big change")
        if ratio > 1.5 or ratio < 0.7: why.append("length")
        if why:
            flags.append((i, ", ".join(why)))
    done = len(sims)
    print(f"[{m}] answered {done}/{len(pairs)} | glossary {gl_ok}/{gl_all} | "
          f"mean similarity to NLLB {sum(sims)/max(done,1):.1f} | mean length ratio {sum(ratios)/max(done,1):.2f}")
    print("   flagged:", flags if flags else "none")
    return flags

all_flags = {m: summarize(m) for m in models}

shown = 0
for i in sorted({i for fl in all_flags.values() for i, _ in fl}):
    if shown == 3:
        break
    print()
    print(f"--- sentence {i} ---")
    print("EN:  ", pairs[i][0])
    print("NLLB:", pairs[i][1])
    for m in models:
        print(f"[{m}]", results[m][i])
    shown += 1

[gemini-3.5-flash] ERROR: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[gemini-3.5-flash] ERROR: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[gemini-3.5-flash] ERROR: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[gemini-3.5-flash] ERROR: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[gemini-3.5-flash] ERROR: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[gemini-3.5-flash] ERROR: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
[gemini-3.5-flash] ERROR: ClientError: 4

In [5]:
import re
from video_to_srt import split_text, fmt_time

CHOSEN = "gemini-3.5-flash"   # or "gemini-3.5-flash-lite", whichever Tamil you prefer

def to_sec(t):
    h, m, rest = t.split(":")
    s, ms = rest.split(",")
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000

def clause_cues(start_str, end_str, ta, max_chars=84):
    pieces = [p for p in re.split(r"(?<=[,;.?!])\s+", ta.strip()) if p]
    packed, cur = [], ""
    for p in pieces:
        if cur and len(cur) + 1 + len(p) > max_chars:
            packed.append(cur)
            cur = p
        else:
            cur = (cur + " " + p).strip()
    if cur:
        packed.append(cur)
    final = []
    for p in packed:
        final += split_text(p, 2) if len(p) > max_chars else [p]
    start, end = to_sec(start_str), to_sec(end_str)
    total = sum(len(p) for p in final)
    out, t = [], start
    for p in final:
        e = t + (end - start) * len(p) / total
        out.append({"start": fmt_time(t), "end": fmt_time(e), "text": p})
        t = e
    return out

final_cues, used_fallback = [], 0
for g, (en, nllb), edited in zip(groups, pairs, results[CHOSEN]):
    text = edited if edited else nllb
    used_fallback += edited is None
    final_cues += clause_cues(en_cues[g[0]]["start"], en_cues[g[-1]]["end"], text)

def write_srt(cs, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, c in enumerate(cs, start=1):
            lines = [c["text"]] if len(c["text"]) <= 42 else split_text(c["text"], 2)
            f.write(f"{i}\n{c['start']} --> {c['end']}\n" + "\n".join(lines) + "\n\n")

write_srt(final_cues, "test_video_tamil_postedit.srt")

def stats(cs):
    durs = [to_sec(c["end"]) - to_sec(c["start"]) for c in cs]
    punct = sum(c["text"].strip()[-1] in ",;.?!" for c in cs)
    return {
        "cues": len(cs),
        "over 84 chars": sum(len(c["text"]) > 84 for c in cs),
        "ending at punctuation": f"{punct}/{len(cs)}",
        "latin letters in cues": sum(bool(re.search(r"[A-Za-z]", c["text"])) for c in cs),
        "shortest (s)": round(min(durs), 2),
        "longest (s)": round(max(durs), 2),
    }

before = parse_srt("test_video_tamil_clauses.srt")
print("model used:", CHOSEN, "| fallbacks to NLLB:", used_fallback)
print("Day 8 (NLLB only):", stats(before))
print("Day 9 (post-edit):", stats(final_cues))

print()
print("sentences that start in lowercase (possible chunk-boundary damage):")
for i, (en, _) in enumerate(pairs):
    if en[0].islower():
        print(f"  sentence {i}: {en}")

print()
for c in final_cues[:6]:
    print(c["start"], "->", c["end"])
    print(c["text"])
    print()

model used: gemini-3.5-flash | fallbacks to NLLB: 12
Day 8 (NLLB only): {'cues': 32, 'over 84 chars': 2, 'ending at punctuation': '26/32', 'latin letters in cues': 2, 'shortest (s)': 0.69, 'longest (s)': 7.25}
Day 9 (post-edit): {'cues': 33, 'over 84 chars': 2, 'ending at punctuation': '27/33', 'latin letters in cues': 1, 'shortest (s)': 0.69, 'longest (s)': 7.25}

sentences that start in lowercase (possible chunk-boundary damage):
  sentence 4: wall, and can discover in it but little of rocky Ithaca.

00:00:00,000 -> 00:00:03,212
திரு. குயில்டர் நடுத்தர வர்க்கத்தின் அப்போஸ்தலர்,

00:00:03,212 -> 00:00:07,080
அவருடைய நற்செய்தியை வரவேற்பதில் நாங்கள் மகிழ்ச்சியடைகிறோம்.

00:00:07,080 -> 00:00:12,280
திரு. குயில்டரின் பாணியும் அவரது விஷயத்தைக் காட்டிலும் குறைவான சுவாரஸ்யமானது அல்ல.

00:00:12,280 -> 00:00:14,137
ஆண்டின் இந்த பண்டிகைக் காலத்தில்,

00:00:14,137 -> 00:00:18,414
கிறிஸ்துமஸும் வறுத்த மாட்டிறைச்சியும் நமக்கு முன்னால் வரவிருக்கும் நிலையில்,

00:00:18,414 -> 00:00:21,903
உண்ணுதலில

In [6]:
import json

with open("glossary.json", "w", encoding="utf-8") as f:
    json.dump(GLOSSARY, f, ensure_ascii=False, indent=1)

a, b = models
rows = []
for i, (en, nllb) in enumerate(pairs):
    reasons = []
    if en[0].islower():
        reasons.append("source starts mid-sentence (chunk-boundary damage)")
    for m in models:
        for j, why in all_flags[m]:
            if j == i:
                reasons.append(f"{m}: {why}")
    if results[a][i] and results[b][i]:
        sim = fuzz.ratio(results[a][i], results[b][i])
        if sim < 70:
            reasons.append(f"the two models disagree (similarity {sim:.0f})")
    if reasons:
        rows.append((i, en, results[CHOSEN][i], reasons))

lines = [f"# Review report ({CHOSEN})", "",
         f"{len(rows)} of {len(pairs)} sentences need a human check.", ""]
for i, en, ta, reasons in rows:
    lines += [f"## Sentence {i}", f"EN: {en}", f"TA: {ta}",
              "Why: " + "; ".join(reasons), ""]

with open("review_report.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("sentences needing review:", len(rows), "of", len(pairs))
for i, en, ta, reasons in rows:
    print(f"  sentence {i}: {', '.join(reasons)}")

sentences needing review: 13 of 14
  sentence 0: gemini-3.5-flash: no result
  sentence 2: the two models disagree (similarity 64)
  sentence 3: gemini-3.5-flash: no result
  sentence 4: source starts mid-sentence (chunk-boundary damage), gemini-3.5-flash-lite: big change, gemini-3.5-flash: no result
  sentence 5: gemini-3.5-flash: no result
  sentence 6: gemini-3.5-flash: no result
  sentence 7: gemini-3.5-flash: no result
  sentence 8: gemini-3.5-flash: no result
  sentence 9: gemini-3.5-flash: no result
  sentence 10: gemini-3.5-flash: no result
  sentence 11: gemini-3.5-flash: no result
  sentence 12: gemini-3.5-flash: no result
  sentence 13: gemini-3.5-flash: no result
